In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from wifiplotting import *
import dill

sequoia = pd.read_csv('../data/sequoia.csv')

TL_CORNER = [37.430582, -122.173904]
BR_CORNER = [37.42705, -122.169413]

In [6]:
osm_context = OSMPlotContext.from_bounds(
    init_lons=[BR_CORNER[1], TL_CORNER[1]], init_lats=[BR_CORNER[0], TL_CORNER[0]],
    pad_fraction=0.0
)

with open('data/osm_context.pkl', 'wb') as f:
    dill.dump(osm_context, f)

In [7]:
sequoia_nonan = sequoia[sequoia.rssi_signal_measured]

scaler = MinMaxScaler()
scaler.fit(pd.DataFrame(np.stack([TL_CORNER[::-1], BR_CORNER[::-1]]), columns=['longitude', 'latitude']))
# scaler.fit(sequoia_nonan[['longitude', 'latitude']])
X_train = pd.DataFrame()
X_train[['longitude', 'latitude']] = scaler.transform(sequoia_nonan[['longitude', 'latitude']])
X_train['indoor'] = sequoia_nonan['indoor'].astype(float).values

y_train = sequoia_nonan['rssi_sample']

X_train = np.array(X_train)
y_train = np.array(y_train)

np.save('data/X_train.npy', X_train)
np.save('data/y_train.npy', y_train)

In [8]:
np.save('data/coord_train.npy', sequoia_nonan[['longitude', 'latitude']].to_numpy())

wlon_train, wlat_train = osm_context.to_world(sequoia_nonan.longitude, sequoia_nonan.latitude)

np.save('data/world_train.npy', np.stack([wlon_train, wlat_train]).T)

In [9]:
grid_width = 100

x_new = np.linspace(0, 1, grid_width)
y_new = np.linspace(0, 1, grid_width)
grid_points = np.stack(np.meshgrid(x_new, y_new), axis=-1).reshape(-1, 2)

coord_grid = scaler.inverse_transform(grid_points)
long_new, lat_new = coord_grid.T
wlon_test, wlat_test = osm_context.to_world(long_new, lat_new)

z_new = osm_context.contains_building(long_new, lat_new).reshape(-1,1)

X_new = np.concatenate([grid_points, z_new], axis=-1)

np.save('data/X_test.npy', X_new)
np.save('data/coord_test.npy', coord_grid)
np.save('data/world_test.npy', np.stack([wlon_test, wlat_test]).T)

In [10]:
X_train.shape

(7796, 3)